[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Tree Shape


## What you will be able to do

Measure how many leaves and how many levels a LightGBM or XGBoost tree has, and set the limits that
decide its shape: `num_leaves` and `max_depth`, and the policy by which a tree grows.

## The idea

### The problem

The **Why Gradient Boosting** notebook gave LightGBM and XGBoost the same 300 trees and the same
learning rate, and the two models still scored differently on the test stations, 2.542 and 2.618.
The size of their trees is one of the differences between them. Unless told otherwise, LightGBM grows
each tree until it has 31 leaves, and XGBoost grows each tree six levels deep. A limit on leaves says
nothing about how deep a tree goes, and a limit on levels says nothing about how many leaves it has,
so the two defaults cannot be compared until each library's trees are measured.

The size of a tree sets which combinations it can describe. A tree three levels deep can follow a
path through three conditions, such as winter and clear and calm, and no more. A tree with many
leaves can carve the table into many small groups, which fits the training stations closely and, past
some point, fits their noise, so the limits on growth decide much of what a model can learn.

### A formal definition

> A tree's shape comes from the limits on its growth. **Depth-wise** growth, XGBoost's default,
> splits every leaf at one level before going to the next, up to **`max_depth`** levels. **Leaf-wise**
> growth, LightGBM's, splits whichever leaf most lowers the loss, wherever it is in the tree, until the
> tree has **`num_leaves`** leaves.

### Why it works that way

A depth-wise tree is balanced. Six levels allow at most 64 leaves, and the tree spends its splits
evenly, so a region of the table that needs many splits gets no more than one that needs few.
A leaf-wise tree spends its splits where the training loss falls most, so it can go deep in one region
of the table and stay shallow elsewhere. Its deep, narrow branches are where it can learn noise from a
few rows, which the **Regularization and Sampling** notebook limits with minimum leaf sizes.

XGBoost can also grow leaf by leaf. With `grow_policy="lossguide"` it grows a tree up to `max_leaves`
leaves, and its default depth limit of 6 still applies until `max_depth` is set to 0. LightGBM always
grows leaf by leaf, and its `max_depth` only caps how deep that growth may go. LightGBM's default
`max_depth` is `-1`, and it reads any value of 0 or below as no limit. XGBoost writes no limit as `0`
and refuses `-1`, which is the first of this notebook's Common errors.

### Where this shows up

The **Early Stopping** notebook found different best counts for the two libraries, and the shapes
this notebook measures are one of the differences behind them. The **Tuning** notebook searches over
these settings, and the **Reading What They Learned** notebook reads the splits these trees make.

### What this notebook covers

- counting the leaves and levels of every tree a model holds
- the shapes the two libraries grow by default
- `num_leaves` and `max_depth` against the validation loss
- XGBoost grown leaf by leaf, and LightGBM limited by depth
- four mistakes, two of which raise nothing

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
import lightgbm as lgb
import xgboost as xgb

rng = np.random.default_rng(0)
x = rng.uniform(0, 1, (1000, 2))
y = np.where(x[:, 0] > 0.75, 10 * x[:, 1], 0.0) + rng.normal(0, 0.1, 1000)

leafwise = lgb.LGBMRegressor(n_estimators=1, num_leaves=8, min_child_samples=5, verbose=-1).fit(x, y)
depthwise = xgb.XGBRegressor(n_estimators=1, max_depth=3).fit(x, y)

nodes = leafwise.booster_.trees_to_dataframe()
print("LightGBM:", nodes["split_feature"].isna().sum(), "leaves, deepest leaf at depth", nodes["node_depth"].max() - 1)
dump = depthwise.get_booster().get_dump()[0].splitlines()
print("XGBoost: ", sum("leaf=" in line for line in dump), "leaves, deepest leaf at depth", max(line.count("\t") for line in dump))
```

```
LightGBM: 8 leaves, deepest leaf at depth 4
XGBoost:  7 leaves, deepest leaf at depth 3
```

The answer changes only where `x[:, 0]` is above 0.75. LightGBM, allowed eight leaves, used all eight
and reached depth 4, while XGBoost, allowed three levels, stopped at depth 3 with seven leaves, one
fewer than the eight three levels allow.

## Setup

The table, the two boosting libraries, and a score.

- `numpy` and `pandas` build the table and count the trees' leaves
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `mean_squared_error` scores a model's predictions

`station_days` makes the same 20,000 made-up station-days as the **Why Gradient Boosting** notebook,
drawn from a generator started from a fixed seed, so the table is the same on every run. The cell
splits the stations into 300 for training, 50 for validation and 50 for testing, keeping all of a
station's days on one side, and then takes each part's eight features as `X_train`, `X_valid` and
`X_test` and its lowest temperatures as `y_train`, `y_valid` and `y_test`.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]
X_train, y_train = train[FEATURES], train["tmin"]
X_valid, y_valid = valid[FEATURES], valid["tmin"]
X_test, y_test = test[FEATURES], test["tmin"]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### Counting the leaves and levels of every tree

Each library can list a model's trees. LightGBM's `trees_to_dataframe` gives a row per node with its
depth, and a leaf is a row with no split feature. XGBoost's `get_dump` gives each tree as text, one line
per node, with a tab for every level below the root and `leaf=` on every leaf.

In [2]:
def lightgbm_shapes(model):
    nodes = model.booster_.trees_to_dataframe()
    leaves = nodes[nodes["split_feature"].isna()]
    return leaves.groupby("tree_index").size(), leaves.groupby("tree_index")["node_depth"].max() - 1


def xgboost_shapes(model):
    trees = [text.splitlines() for text in model.get_booster().get_dump()]
    leaves = pd.Series([sum("leaf=" in line for line in lines) for lines in trees])
    depths = pd.Series([max(line.count("\t") for line in lines) for lines in trees])
    return leaves, depths


def describe(name, shapes):
    leaves, depths = shapes
    print(f"{name}: {leaves.mean():.1f} leaves on average, {leaves.max()} at most | "
          f"depth {depths.mean():.1f} on average, {depths.max()} at most")

### The shapes each library grows by default

In [3]:
lightgbm_default = lgb.LGBMRegressor(n_estimators=100, random_state=0, verbose=-1).fit(X_train, y_train)
xgboost_default = xgb.XGBRegressor(n_estimators=100, random_state=0).fit(X_train, y_train)

describe("LightGBM", lightgbm_shapes(lightgbm_default))
describe("XGBoost ", xgboost_shapes(xgboost_default))

LightGBM: 31.0 leaves on average, 31 at most | depth 10.0 on average, 14 at most
XGBoost : 57.6 leaves on average, 64 at most | depth 6.0 on average, 6 at most


LightGBM's trees all have 31 leaves, its limit, and reach from 7 to 14 levels deep, 10 on average.
XGBoost's are all six levels deep and hold 57.6 leaves on average, short of the 64 that six levels
allow wherever a node was left unsplit.

In [4]:
leaves, depths = lightgbm_shapes(lightgbm_default)
print("LightGBM trees by depth:", depths.value_counts().sort_index().to_dict())
print("deeper than 10 levels:", int((depths > 10).sum()), "of", len(depths))

LightGBM trees by depth: {7: 6, 8: 25, 9: 21, 10: 11, 11: 8, 12: 14, 13: 9, 14: 6}
deeper than 10 levels: 37 of 100


Leaf-wise growth takes a tree as deep as its best splits lead, and 37 of LightGBM's 100 trees go
deeper than ten levels, where every XGBoost tree stops at six.

### num_leaves against the validation loss

Each model below is stopped early on the validation stations, at a learning rate of 0.1, so that each
tree size gets the number of trees that suits it.

In [5]:
for leaves in (4, 8, 31, 127):
    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, num_leaves=leaves, random_state=0, verbose=-1)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
    print(f"num_leaves {leaves:3}: best {model.best_iteration_:4} trees, validation {model.best_score_['valid_0']['l2']:.3f}")

num_leaves   4: best  534 trees, validation 2.629
num_leaves   8: best  493 trees, validation 2.352
num_leaves  31: best  120 trees, validation 2.546
num_leaves 127: best   83 trees, validation 2.638


Eight leaves give the lowest validation loss, 2.352, after 493 trees. Four leaves need more trees and
end higher, and 31 and 127 leaves stop early, after 120 and 83 trees, at 2.546 and 2.638. The four
numbers come from one split and one seed, and the **Tuning** notebook searches with cross-validation
instead.

### max_depth against the validation loss

In [6]:
for depth in (2, 3, 6, 10):
    model = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=depth, early_stopping_rounds=100,
                             random_state=0)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    print(f"max_depth {depth:2}: best {model.best_iteration + 1:4} trees, "
          f"validation {mean_squared_error(y_valid, model.predict(X_valid)):.3f}")

max_depth  2: best 1215 trees, validation 2.765
max_depth  3: best  435 trees, validation 2.629
max_depth  6: best   77 trees, validation 2.526
max_depth 10: best   66 trees, validation 2.884


For XGBoost, six levels give the lowest validation loss of the four, 2.526 after 77 trees, and two
levels need 1,215 trees to reach only 2.765. A shallow tree combines fewer conditions, so a model of
shallow trees needs more of them, and the deepest, ten levels, stops after 66 trees at 2.884. These
four also come from one split, and three levels, 0.103 behind, are close enough that another split of
the stations could choose them.

### XGBoost grown leaf by leaf

`grow_policy="lossguide"` makes XGBoost grow leaf-wise, `max_leaves=31` gives it LightGBM's default
limit, and `max_depth=0` removes the depth limit, which is XGBoost's way of writing "none".

In [7]:
lossguide = xgb.XGBRegressor(n_estimators=100, grow_policy="lossguide", max_leaves=31, max_depth=0, random_state=0)
lossguide.fit(X_train, y_train)
describe("XGBoost, leaf by leaf", xgboost_shapes(lossguide))
describe("LightGBM, default    ", lightgbm_shapes(lightgbm_default))

XGBoost, leaf by leaf: 31.0 leaves on average, 31 at most | depth 10.3 on average, 15 at most
LightGBM, default    : 31.0 leaves on average, 31 at most | depth 10.0 on average, 14 at most


Grown leaf by leaf with 31 leaves, XGBoost's trees take nearly the shape of LightGBM's, 31 leaves and
about ten levels on average, 15 at most against LightGBM's 14.

### LightGBM limited by depth

`max_depth=4` caps LightGBM's trees at four levels, and `num_leaves=16`, the most four levels hold,
sets the leaf limit to match, so neither limit is left unused.

In [8]:
shallow = lgb.LGBMRegressor(n_estimators=100, max_depth=4, num_leaves=16, random_state=0, verbose=-1)
shallow.fit(X_train, y_train)
describe("LightGBM, depth 4", lightgbm_shapes(shallow))

LightGBM, depth 4: 15.6 leaves on average, 16 at most | depth 4.0 on average, 4 at most


Capped at four levels with room for 16 leaves, LightGBM's trees fill nearly all of them, 15.6 on
average, which is the shape of a depth-wise tree four levels deep.

### The chosen shapes on the test stations

The validation stations chose eight leaves for LightGBM and six levels for XGBoost above. Each is trained
again with that shape and stopped on the validation stations, and the test stations are measured once.

In [9]:
lightgbm_chosen = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, num_leaves=8, random_state=0, verbose=-1)
lightgbm_chosen.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
xgboost_chosen = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=6, early_stopping_rounds=100,
                                  random_state=0)
xgboost_chosen.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

print("LightGBM, 8 leaves:", round(mean_squared_error(y_test, lightgbm_chosen.predict(X_test)), 3))
print("XGBoost, depth 6:  ", round(mean_squared_error(y_test, xgboost_chosen.predict(X_test)), 3))

LightGBM, 8 leaves: 2.584
XGBoost, depth 6:   2.614


On the test stations the LightGBM model with eight leaves scores 2.584 and XGBoost with six levels
2.614. Eight leaves had the lowest validation loss of LightGBM's four settings, 2.352, and part of
that came from the particular validation stations, which the test stations do not share. A
validation score that won a comparison flatters its model, and the test stations, measured once, do
not.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/06-tree-shape-solutions.ipynb).

**1.** Train LightGBM with `num_leaves=15`, stopped early on the validation stations at a learning rate of 0.1, and print its best iteration and validation loss beside the sweep's.

In [10]:
# your code here

**2.** Train 100 XGBoost trees with `max_depth=4`, and print their shapes with `describe` and `xgboost_shapes`.

In [11]:
# your code here

**3.** Train XGBoost leaf by leaf with `max_leaves=8` and no depth limit, stopped early at a learning rate of 0.1, and print its validation loss beside LightGBM's with `num_leaves=8`.

In [12]:
# your code here

**4.** Count how many of `lightgbm_chosen`'s trees reach seven levels, the most a tree with eight leaves can have.

In [13]:
# your code here

## Common errors

### `XGBoostError: value -1 for Parameter max_depth should be greater equal to 0`

LightGBM writes "no depth limit" as `-1`, and a program that moves from one library to the other
carries the value over.

In [14]:
xgb.XGBRegressor(n_estimators=100, max_depth=-1).fit(X_train, y_train)

XGBoostError: value -1 for Parameter max_depth should be greater equal to 0
max_depth: Maximum depth of the tree; 0 indicates no limit; a limit is required for depthwise policy

XGBoost's limit must be 0 or more, and the message's second line says that 0 means no limit and that
depth-wise growth needs a limit. Give XGBoost a depth such as 6, or `max_depth=0` together with
`grow_policy="lossguide"` and a `max_leaves`, as the worked examples do.

### No error, and trees with thousands of leaves

In [15]:
unlimited = xgb.XGBRegressor(n_estimators=20, max_depth=0, random_state=0).fit(X_train, y_train)
limited = xgb.XGBRegressor(n_estimators=20, random_state=0).fit(X_train, y_train)

describe("max_depth=0", xgboost_shapes(unlimited))
for name, model in (("max_depth=0", unlimited), ("max_depth=6", limited)):
    print(f"{name}: training {mean_squared_error(y_train, model.predict(X_train)):.3f}, "
          f"validation {mean_squared_error(y_valid, model.predict(X_valid)):.3f}")

max_depth=0: 5900.0 leaves on average, 7934 at most | depth 30.4 on average, 36 at most
max_depth=0: training 0.003, validation 3.673
max_depth=6: training 1.689, validation 2.898


`max_depth=0` raises nothing, although the last message said depth-wise growth needs a limit. The 20
trees grew to 5,900 leaves and 30 levels on average, and 15,000 training rows spread over 5,900 leaves
is about two and a half rows a leaf. They fit the training stations almost exactly, at 0.003, and do
worse on the validation stations than 20 trees of depth 6, 3.673 against 2.898. Use `max_depth=0`
only with `grow_policy="lossguide"` and a `max_leaves`.

### No error, and a depth limit that leaves every tree at 31 leaves

In [16]:
default = lgb.LGBMRegressor(n_estimators=100, random_state=0, verbose=-1).fit(X_train, y_train)
capped = lgb.LGBMRegressor(n_estimators=100, max_depth=12, random_state=0, verbose=-1).fit(X_train, y_train)
describe("max_depth=12", lightgbm_shapes(capped))
describe("default     ", lightgbm_shapes(default))

max_depth=12: 31.0 leaves on average, 31 at most | depth 9.7 on average, 12 at most
default     : 31.0 leaves on average, 31 at most | depth 10.0 on average, 14 at most


`max_depth=12` cut the deepest trees from 14 levels to 12 and left every tree with its 31 leaves.
`num_leaves` is the limit that sets a LightGBM tree's size, and a depth cap matters only when it is
below the depth 31 leaves reach. Set both, as the worked example with `max_depth=4` and
`num_leaves=16` does.

### `LightGBMError: Parameter num_leaves should be of type int, got "32.0"`

A program works out the number of leaves from a depth, and division gives a float.

In [17]:
depth = 5
lgb.LGBMRegressor(n_estimators=100, num_leaves=2 ** (depth + 1) / 2, verbose=-1).fit(X_train, y_train)

[LightGBM] [Fatal] Parameter num_leaves should be of type int, got "32.0"


LightGBMError: Parameter num_leaves should be of type int, got "32.0"

LightGBM's C++ code checks the type of every setting, prints the problem on a line of its own marked
`[Fatal]`, and raises. `2 ** (depth + 1) / 2` is 32.0, a float, because `/` in Python always gives one.
Use `//` or `int(...)`, or write `2 ** depth`.

## Recap

- You can count the leaves and levels of every tree in a LightGBM or XGBoost model.
- You can tell depth-wise growth, XGBoost's default, from leaf-wise growth, LightGBM's.
- You can set `num_leaves` and `max_depth` and let the validation stations choose between shapes.
- You can make XGBoost grow leaf by leaf and cap LightGBM's depth, writing "no limit" as each library
  expects.

## What is next

The **Regularization and Sampling** notebook limits trees in other ways: penalties on their leaves'
values, minimum leaf sizes, and fitting each tree on a sample of the rows or columns.

---

&#8592; **Previous:** [Early Stopping](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/05-early-stopping.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Regularization and Sampling](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/07-regularization-and-sampling.ipynb) &#8594;
